# Functional Regression: Input Time Series (11 pts + 3 params) → Output Time Series (1500 pts)

**Pipeline:**  
1. Feature-engineer the 11-pt input time series (raw values + first differences + cumulative sum)  
2. Combine with 3 scalar parameters → 35-feature vector  
3. Compress the 1500-pt output with PCA (retain 95% variance)  
4. Train `GradientBoostingRegressor` wrapped in `MultiOutputRegressor` on PCA scores  
5. Reconstruct full output via inverse PCA  
6. Optional: smooth output curves with UnivariateSpline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

from scipy.interpolate import UnivariateSpline

# --- Deep learning (activate later) ---
# import torch
# import torch.nn as nn
# from torch.utils.data import DataLoader, TensorDataset

## Data
Replace the placeholder arrays with real data. Keep the shape conventions:
- `X_series`: `(N, 11)` — input time series, one row per sample
- `X_params`: `(N, 3)` — scalar conditioning parameters
- `Y_raw`: `(N, 1500)` — target output time series

In [ ]:
N = 8  # number of samples  <-- update when you have real data

# Time axes (adjust step / span to match your data)
INPUT_TIME  = np.linspace(0, 1, 11)    # 11-pt input time axis
OUTPUT_TIME = np.linspace(0, 1, 1500)  # 1500-pt output time axis

# --- PLACEHOLDER DATA — replace with real arrays ---
np.random.seed(42)
X_series = np.random.rand(N, 11)   # (N, 11) input time series   <-- replace
X_params = np.random.rand(N, 3)    # (N, 3)  scalar parameters   <-- replace
Y_raw    = np.random.rand(N, 1500) # (N, 1500) output curves     <-- replace
# ---------------------------------------------------

print(f"X_series : {X_series.shape}")
print(f"X_params : {X_params.shape}")
print(f"Y_raw    : {Y_raw.shape}")

## Input Feature Engineering
Augment the 11-pt time series with temporal features to expose its shape to the model:

| Group | Size | Captures |
|---|---|---|
| Raw values | 11 | Amplitude at each time step |
| First differences (Δ) | 10 | Rate of change |
| Cumulative sum | 11 | Integral / trend |
| Scalar params | 3 | Conditioning |
| **Total** | **35** | |

In [ ]:
diffs  = np.diff(X_series, axis=1)       # (N, 10)
cumsum = np.cumsum(X_series, axis=1)     # (N, 11)
X_feat = np.hstack([X_series, diffs, cumsum, X_params])  # (N, 35)

print(f"X_feat shape: {X_feat.shape}  (expected: ({N}, 35))")

## Train / Test Split
Split **before** fitting any scaler or PCA to avoid data leakage.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X_feat, Y_raw, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} samples  |  Test: {X_test.shape[0]} samples")

## Normalization Scaffold
`scaler_X` is active now and is **required** for deep learning later.  
`scaler_Y` is commented out — **activate** when switching to a neural network.

In [ ]:
# Input normalization (used by both ML and DL)
scaler_X = StandardScaler()
X_scaled_train = scaler_X.fit_transform(X_train)
X_scaled_test  = scaler_X.transform(X_test)

# Output normalization — activate for deep learning
# scaler_Y = StandardScaler()
# Y_scaled_train = scaler_Y.fit_transform(Y_train)
# Y_scaled_test  = scaler_Y.transform(Y_test)

## PCA on Output
Compresses the 1500-dimensional output to `ky` principal components (95% variance retained).  
This reduces the effective regression target from 1500 scalars to ~30–80, greatly improving accuracy on small N.

In [ ]:
pca_y = PCA(n_components=0.95)
Z_y_train = pca_y.fit_transform(Y_train)   # (N_train, ky)
Z_y_test  = pca_y.transform(Y_test)        # (N_test,  ky)

ky = pca_y.n_components_
print(f"Output PCA: {ky} components retain 95% variance (from 1500)")

# Scree plot
cumvar = np.cumsum(pca_y.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(np.arange(1, ky + 1), cumvar * 100, marker='o', markersize=3)
ax.axhline(95, color='red', linestyle='--', label='95% threshold')
ax.set_xlabel('Number of components')
ax.set_ylabel('Cumulative variance explained (%)')
ax.set_title('Output PCA — Scree plot')
ax.legend()
plt.tight_layout()
plt.show()

## ML Model: GradientBoosting + MultiOutputRegressor
Maps the 35-feature input vector to the `ky` PCA scores of the output.  
Output is then reconstructed via `pca_y.inverse_transform`.

In [ ]:
base_estimator = GradientBoostingRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    min_samples_leaf=2,
    random_state=42
)

model = MultiOutputRegressor(base_estimator, n_jobs=-1)
model.fit(X_scaled_train, Z_y_train)

Z_pred = model.predict(X_scaled_test)              # (N_test, ky)
Y_pred = pca_y.inverse_transform(Z_pred)           # (N_test, 1500)

print(f"Y_pred shape: {Y_pred.shape}")

## Optional: Smooth Predicted Output Curves
Applies a cubic smoothing spline to each predicted curve.  
Tune `S_PARAM`: smaller → tighter fit, larger → smoother curve.

In [ ]:
S_PARAM = 0.01  # smoothing parameter — tune to your data

Y_pred_smooth = np.empty_like(Y_pred)
for i in range(len(Y_pred)):
    spl = UnivariateSpline(OUTPUT_TIME, Y_pred[i], k=3, s=S_PARAM)
    Y_pred_smooth[i] = spl(OUTPUT_TIME)

## Evaluation

In [ ]:
mse_ml  = mean_squared_error(Y_test, Y_pred)
r2_ml   = r2_score(Y_test, Y_pred)
mse_spl = mean_squared_error(Y_test, Y_pred_smooth)
r2_spl  = r2_score(Y_test, Y_pred_smooth)

print(f"ML prediction  — MSE: {mse_ml:.6f}  R²: {r2_ml:.4f}")
print(f"Spline smoothed — MSE: {mse_spl:.6f}  R²: {r2_spl:.4f}")

# --- Visual comparison for one sample ---
SAMPLE_IDX = 0  # change to inspect different test samples

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Input time series
ax = axes[0]
ax.plot(INPUT_TIME, X_test[SAMPLE_IDX, :11], marker='o', label='Input series (11 pts)')
ax.set_title(f'Sample {SAMPLE_IDX} — Input time series')
ax.set_xlabel('Input time')
ax.legend()
ax.grid(True, alpha=0.3)

# Output: true vs predicted
ax = axes[1]
ax.plot(OUTPUT_TIME, Y_test[SAMPLE_IDX],         label='True output',      linewidth=2)
ax.plot(OUTPUT_TIME, Y_pred[SAMPLE_IDX],         label='ML prediction',    linewidth=1.5, linestyle='--')
ax.plot(OUTPUT_TIME, Y_pred_smooth[SAMPLE_IDX],  label='Spline smoothed',  linewidth=1.5, linestyle=':')
ax.set_title(f'Sample {SAMPLE_IDX} — Output time series (1500 pts)')
ax.set_xlabel('Output time')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Deep Learning — Placeholder (commented out)
When ready to switch to a neural network:
1. Activate `scaler_Y` in the normalization cell above
2. Uncomment and implement the network below
3. Keep `scaler_X` and `pca_y` — they remain useful for DL too

In [ ]:
# TODO: replace ML model with a PyTorch / Keras network
#
# Option A — predict PCA scores (keeps inverse_transform step):
#   Input:  X_scaled_train  (N, 35)
#   Target: Z_y_train       (N, ky)   ← pca_y.inverse_transform to get back 1500 pts
#
# Option B — predict full output directly (activate scaler_Y above):
#   Input:  X_scaled_train  (N, 35)
#   Target: Y_scaled_train  (N, 1500) ← scaler_Y.inverse_transform to get real units
#
# Option C — use 1D-CNN / LSTM encoder on raw input series:
#   Input:  X_series (N, 11) fed as sequences + X_params (N, 3) as conditioning
#   Target: Z_y_train (N, ky)  or  Y_scaled_train (N, 1500)
#
# import torch
# import torch.nn as nn
# ...
pass